# Magenta RealTime 2 on Kaggle (2x T4)


Self-contained, copy-paste setup that: pins a non-drifting JAX/CUDA stack,
downloads resources + the mrt2_base checkpoint, quantizes it to bf16, detects
the GPUs, shards the 2.4B model across 2 T4s (tensor parallelism) when 2 GPUs
are present (falling back to a single T4 otherwise), and generates 8 s of audio.

**Run on a Kaggle notebook with GPU accelerator enabled (2x T4).**
All GPU execution happens here on Kaggle; nothing in this notebook is run on
the dev machine it was authored on.


## 1. JAX memory defaults (set BEFORE importing jax)


In [ ]:
# These must be set before `import jax` so jaxlib reads them at backend init.
# preallocate=false avoids BFC fragmentation OOMs when JAX shares a T4 with
# the MusicCoCa (TFLite) interpreter. Override freely; these are safe defaults.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.85')
# Keep downloads on /kaggle/working (writable, persistent, more space).
os.environ.setdefault('MAGENTA_HOME', '/kaggle/working/magenta-rt-v2')
print('MAGENTA_HOME =', os.environ['MAGENTA_HOME'])


## 2. Clone the repo (with the vendored sequence-layers submodule)


In [ ]:
%%bash
cd /kaggle/working
# Clone YOUR fork (push the patched repo to GitHub first), then install from it.
# Override by setting MRT_REPO, e.g.:
#   MRT_REPO=https://github.com/<you>/magenta-realtime.git
MRT_REPO=${MRT_REPO:-https://github.com/magenta/magenta-realtime.git}
if [ ! -d magenta-realtime ]; then
  git clone --recurse-submodules "$MRT_REPO" magenta-realtime
else
  cd magenta-realtime && git pull --quiet --recurse-submodules 2>/dev/null || true
fi


## 3. Install a non-drifting JAX/CUDA stack + the repo


In [ ]:
# Install the pinned GPU stack from requirements-kaggle.txt, then the repo
# with --no-deps so the project's [jax] extra can NEVER reinstall one of the
# four cuda packages (jax/jaxlib/jax-cuda12-plugin/jax-cuda12-pjrt) out of sync.
%%bash
cd /kaggle/working/magenta-realtime
pip install -q -r requirements-kaggle.txt
pip install -q -e . --no-deps


## 4. Verify the four CUDA packages are the SAME version (no silent drift)


In [ ]:
import importlib.metadata as m
for pkg in ['jax','jaxlib','jax-cuda12-plugin','jax-cuda12-pjrt']:
    try:
        print(f'{pkg:22s} {m.version(pkg)}')
    except Exception as e:
        print(f'{pkg:22s} MISSING ({e})')
import numpy, numba
print('numpy', numpy.__version__, '| numba', numba.__version__)


## 5. Download MusicCoCa/SpectroStream resources + the mrt2_base checkpoint


In [ ]:
# Positional NAME (not --model=...). Saved under $MAGENTA_HOME.
%%bash
cd /kaggle/working/magenta-realtime
mrt models init
mrt checkpoints download mrt2_base


## 6. Quantize the checkpoint to bf16 (halves memory: 9.84 GB -> 4.92 GB)


In [ ]:
# Runs on CPU (numpy + ml_dtypes). The model already computes in bfloat16, so
# this only changes storage/on-GPU param memory, not numerics. Using the bf16
# checkpoint means a single T4 fits mrt2_base, and 2x T4 sharding has huge
# headroom. (Safe to skip if you prefer fp32 + 2-GPU sharding.)
%%bash
cd /kaggle/working/magenta-realtime
mrt checkpoints quantize mrt2_base --dtype bf16


## 7. Detect GPUs and assert CUDA is actually available (no silent CPU fallback)


In [ ]:
import logging, jax
logging.basicConfig(level=logging.INFO, force=True)
from magenta_rt.jax import _gpu_check
print(_gpu_check.diagnose_devices())
_gpu_check.assert_gpu_available()   # raises loudly if JAX fell back to CPU
n_gpu = _gpu_check.num_local_cuda_devices()
print(f'\nCUDA devices detected: {n_gpu}')
print('Mode:', 'sharded (2 GPUs)' if n_gpu >= 2 else 'single-GPU')


## 8. Build the system (shard across 2 GPUs if available) and generate 8 s


In [ ]:
# shard=True shards the 2.4B model across all local CUDA GPUs via tensor
# parallelism (Megatron-style: model dim + heads on the 'model' mesh axis).
# Falls back to single-GPU automatically if <2 GPUs or sharding setup fails.
from magenta_rt import MagentaRT2Jax
from magenta_rt.config import MUSICCOCA

CKPT = 'mrt2_base_bf16.safetensors'  # the bf16 checkpoint from step 6
mrt = MagentaRT2Jax(
    size='mrt2_base',
    checkpoint=CKPT,
    shard=(n_gpu >= 2),               # opt into 2-GPU sharding
    require_gpu=True,
)
print('Sharded:', mrt._sharded, '| mesh:', mrt._mesh)


In [ ]:
embedding = mrt.embed_style('disco funk', use_mapper=True)
frames = 8 * 25  # 8 seconds at 25 Hz
wav, state = mrt.generate(conditioning={MUSICCOCA.key: embedding}, frames=frames)


## 9. Save + play the result, and print per-device memory


In [ ]:
import IPython.display as ipd
out = '/kaggle/working/mrt2_base_output.wav'
wav.write(out)
print('Saved', out, f'({wav.sample_rate} Hz, {len(wav.samples)/wav.sample_rate:.1f}s)')
ipd.display(ipd.Audio(wav.samples.T, rate=wav.sample_rate))


In [ ]:
# Final per-device memory report (best-effort, CUDA only).
import jax
for d in jax.devices():
    if d.platform != 'gpu':
        continue
    s = d.memory_stats() or {}
    used = s.get('bytes_in_use', 0) / 1e9
    lim = (s.get('limit') or s.get('bytes_limit') or 0) / 1e9
    print(f'{d}: {used:.2f} GB / {lim:.2f} GB in use')
